In [ ]:
!nvidia-smi
%pip install -q "diffusers>=0.33" transformers accelerate ftfy imageio imageio-ffmpeg sentencepiece

In [ ]:
import torch
from diffusers import AutoencoderKLWan, WanPipeline

MODEL = 'Wan-AI/Wan2.1-T2V-1.3B-Diffusers'
vae = AutoencoderKLWan.from_pretrained(MODEL, subfolder='vae', torch_dtype=torch.float32)
pipe = WanPipeline.from_pretrained(MODEL, vae=vae, torch_dtype=torch.float16)
# UMT5-XXL (~13GB fp16) OOMs a 15GB T4 under model offload — stream modules instead
pipe.enable_sequential_cpu_offload()

In [ ]:
import json, os
from pathlib import Path
from diffusers.utils import export_to_video

spec = json.loads(r'''{"comment": "LaunchGrid human b-roll shot list \u2014 fed to generate_wan_colab.ipynb. Portrait 480x832 (upscale later). Concepts marked \u2605 in VIDEO_ADS_50.md (single-scene, no lip-sync, no on-screen text \u2014 ideal for open video models). Keep prompts atmosphere-driven; UI/text gets composited crisply in Remotion, never generated.", "defaults": {"width": 480, "height": 832, "num_frames": 81, "fps": 16, "steps": 30, "guidance": 5.0}, "negative": "text, watermark, logo, subtitles, distorted hands, extra fingers, deformed face, low quality, jpeg artifacts, cartoon, anime", "clips": [{"id": "ping-1143", "prompt": "Cinematic close-up, dark Indian bedroom at night, a smartphone on a wooden bedside table lights up with a soft warm glow, a young woman's face is gently illuminated as she wakes and smiles faintly, shallow depth of field, warm orange screen light against cool blue moonlight, photorealistic, film grain"}]}''')
os.makedirs('out', exist_ok=True)
d = spec['defaults']
for clip in spec['clips']:
    dst = f"out/{clip['id']}.mp4"
    if Path(dst).exists():
        print('skip', dst); continue
    print('generating', clip['id'], '...', flush=True)
    frames = pipe(
        prompt=clip['prompt'],
        negative_prompt=spec['negative'],
        width=clip.get('width', d['width']),
        height=clip.get('height', d['height']),
        num_frames=clip.get('num_frames', d['num_frames']),
        num_inference_steps=clip.get('steps', d['steps']),
        guidance_scale=clip.get('guidance', d['guidance']),
        generator=torch.Generator('cuda').manual_seed(clip.get('seed', 42)),
    ).frames[0]
    export_to_video(frames, dst, fps=clip.get('fps', d['fps']))
    print('wrote', dst, flush=True)
print('ALL DONE')